In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader

from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

from dotenv import load_dotenv
import os

In [ ]:
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash")

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8680.37it/s]


In [8]:
pdf_link = "data/edital-mestrado.pdf"
loader = PyPDFLoader(pdf_link, extract_images=False)
pages = loader.load_and_split()
len(pages)

15

In [ ]:
# splitter
child_splitter = RecursiveCharacterTextSplitter(chunk_size=200)
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000,
    chunk_overlap=200,
    length_function=len,
    add_start_index=True
)

In [11]:
# storages
store = InMemoryStore()
vectorstore = Chroma(embedding_function=embedding_model, persist_directory="childVectorDB")

/var/folders/p9/pwvnb57s6211pm7jb8rn0q480000gn/T/ipykernel_24121/3227047581.py:3: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(embedding_function=embedding_model, persist_directory="childVectorDB")


In [12]:
parent_document_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter
)

parent_document_retriever.add_documents(pages, ids=None)

In [13]:
parent_document_retriever.vectorstore.get()

{'ids': ['7e7b7250-86a3-43ee-bde0-dbee46e0f203',
  '3cc9298a-39cd-45a8-921e-9243b4d27ef8',
  'f9371f9d-c5b8-4ed6-94f4-16357a7a87ef',
  '12043f09-edbc-4f5a-a5a1-9dc231c1c9af',
  'ede76fa4-b57c-4563-b518-a1c7dcfe7787',
  '3f56e5b8-b715-4d9b-973c-d788296bcdb1',
  '81713af9-3295-498a-a890-4834149c3b4c',
  '75a13d1c-efd0-4943-bcb4-6eb34d737581',
  'e2f1ef88-09c5-4f31-a3ca-4a4c11fe83d0',
  '35a81aeb-55b3-49de-b2ef-6addc680d158',
  '0553565e-6e64-4fa6-9bad-fbf4ff9452bf',
  '0f1a659f-5554-42e0-ab83-63e3e15053e2',
  'df1bc461-d38f-43e4-8c01-7e96328a6dca',
  'c49628d7-e18b-47d4-a009-da1af06547fc',
  '33dd673e-c712-402d-8593-fe45f9d1cc49',
  '8307479e-4e04-4888-8065-ebb8f07b6ec7',
  '93fc46e3-4c32-4bae-ac14-d253c4be334c',
  'db5023bf-47bc-4984-a987-ae0edd923b04',
  '6af31dd8-8ba7-41b7-bcc4-abbf383178ba',
  'bea11db8-dd57-46fa-981c-a7a2e2622916',
  'd195be20-70dc-4fa2-8679-67b1b3bfb519',
  '3e757674-eff5-4136-89f1-256efe1a70a1',
  'ff8e3000-5df9-4e3b-adb4-1df38e4e4d1d',
  '855ba2b5-053c-49e8-a8fe-

In [21]:
TEMPLATE = """
    Você é um secretário da coordenação de pós-graduação da UFCG. Responda a pergunta abaixo utilizando o contexto informado.
    Query:
    {question}

    Context:
    {context}
"""

rag_prompt = ChatPromptTemplate.from_template(TEMPLATE)

In [19]:
setup_retrieval = RunnableParallel({"question": RunnablePassthrough(), "context": parent_document_retriever})

output_parser = StrOutputParser() # transforma a resposta do modelo em uma string simples 

In [22]:
# a saída de uma etapa vira entrada da próxima com o operador |
parent_chain_retrieval = setup_retrieval | rag_prompt | llm | output_parser

In [ ]:
parent_chain_retrieval.invoke("Como é considerado a pontuação do aluno pelo desempenho acadêmico?")

'Olá! Como secretário da Coordenação de Pós-Graduação, informo que a pontuação e avaliação do desempenho acadêmico do aluno/candidato é considerada segundo os critérios estabelecidos no edital:\n\n1. **Requisito Mínimo de Habilitação:**\n   Para estar habilitado no processo seletivo, o candidato precisa apresentar histórico escolar (de graduação para mestrado ou de mestrado para doutorado) com CRA, IRA ou índice institucional equivalente **maior ou igual a 6,5**.\n\n2. **Forma de Cálculo do Índice:**\n   O cálculo do índice de desempenho acadêmico (CRA, IRA ou equivalente) considera:\n   * **$Ni$:** a nota obtida em cada disciplina $i$;\n   * **$ci$:** o número de horas (carga horária) dessas disciplinas;\n   * **$Nm$:** a quantidade total de disciplinas em que o aluno se matriculou ou solicitou dispensa e possui nota aferida.\n   * *Atenção:* Disciplinas trancadas ou dispensadas sem nota **não** são computadas no cálculo.\n\n3. **Inclusão na Avaliação do Processo Seletivo:**\n   * O v